Phase 1 – Document Ingestion
Step 1 – Install Libraries

In [25]:
!pip install -q boto3 pymupdf pandas numpy faiss-cpu

Step 2 – Verify the Environment

In [26]:
import boto3
import fitz
import numpy as np
import pandas as pd

print("=" * 50)
print("Environment Verification")
print("=" * 50)

print("Boto3 Version   :", boto3.__version__)
print("NumPy Version   :", np.__version__)
print("Pandas Version  :", pd.__version__)
print("PyMuPDF Version :", fitz.__doc__.split()[1])

print("\n✅ All libraries imported successfully!")

Environment Verification
Boto3 Version   : 1.43.36
NumPy Version   : 2.2.6
Pandas Version  : 2.3.3
PyMuPDF Version : 1.28.0:

✅ All libraries imported successfully!


Step 3 – Verify AWS Identity

In [27]:
import boto3

sts = boto3.client("sts")

identity = sts.get_caller_identity()

print("=" * 50)
print("AWS Identity")
print("=" * 50)

print("Account ID :", identity["Account"])
print("ARN        :", identity["Arn"])

AWS Identity
Account ID : 334590195171
ARN        : arn:aws:sts::334590195171:assumed-role/genaiapp/SageMaker


Cell 1 – Define Configuration

In [28]:
import boto3
import os

# AWS Configuration
BUCKET_NAME = "edi-documents-ajam-2026"
PDF_FILE = "Kaveri_English_Text_Book_Class_9.pdf"

# Local project path
LOCAL_PATH = "../data/pdf/Kaveri_English_Text_Book_Class_9.pdf"

print("Bucket :", BUCKET_NAME)
print("PDF    :", PDF_FILE)
print("Local  :", LOCAL_PATH)

Bucket : edi-documents-ajam-2026
PDF    : Kaveri_English_Text_Book_Class_9.pdf
Local  : ../data/pdf/Kaveri_English_Text_Book_Class_9.pdf


Cell 2 – Download the PDF

In [29]:
import boto3

s3 = boto3.client("s3")

print("Downloading PDF from S3...")

s3.download_file(
    BUCKET_NAME,
    PDF_FILE,
    LOCAL_PATH
)

print("✅ Download Complete")

✅ Download Complete


Cell 3 – Verify the File

In [30]:
import os

size_mb = os.path.getsize(LOCAL_PATH) / (1024 * 1024)

print("=" * 50)
print("Downloaded File Verification")
print("=" * 50)

print("File Exists :", os.path.exists(LOCAL_PATH))
print("Size (MB)   :", round(size_mb, 2))

Downloaded File Verification
File Exists : True
Size (MB)   : 176.34


Cell 4 — Open the PDF

In [31]:
import fitz

doc = fitz.open(LOCAL_PATH)

print("=" * 50)
print("PDF Opened Successfully")
print("=" * 50)

print("Total Pages :", len(doc))

PDF Opened Successfully
Total Pages : 300


Cell 5 — Read the First 10 Pages

In [32]:
pages = []

TOTAL_PAGES_TO_READ = 10

for page_no in range(min(TOTAL_PAGES_TO_READ, len(doc))):

    page = doc.load_page(page_no)

    text = page.get_text("text")

    images = page.get_images(full=True)

    pages.append({
        "page": page_no + 1,
        "text": text,
        "images": images
    })

print(f"Successfully read {len(pages)} pages.")

Successfully read 10 pages.


Cell 6 — Verify Extraction

In [33]:
print("=" * 60)
print("Extraction Summary")
print("=" * 60)

for p in pages:

    print(f"Page : {p['page']}")
    print(f"Characters : {len(p['text'])}")
    print(f"Images : {len(p['images'])}")

    print("-" * 60)

Extraction Summary
Page : 1
Characters : 0
Images : 1
------------------------------------------------------------
Page : 2
Characters : 0
Images : 1
------------------------------------------------------------
Page : 3
Characters : 39
Images : 2
------------------------------------------------------------
Page : 4
Characters : 2101
Images : 5
------------------------------------------------------------
Page : 5
Characters : 2451
Images : 6
------------------------------------------------------------
Page : 6
Characters : 2077
Images : 9
------------------------------------------------------------
Page : 7
Characters : 2152
Images : 10
------------------------------------------------------------
Page : 8
Characters : 2554
Images : 9
------------------------------------------------------------
Page : 9
Characters : 2445
Images : 10
------------------------------------------------------------
Page : 10
Characters : 2521
Images : 9
---------------------------------------------------------

Cell 7 — Preview the Text

In [34]:
print("=" * 60)
print("Preview of First Page with Text")
print("=" * 60)

for p in pages:

    if len(p["text"].strip()) > 100:

        print(f"Page Number : {p['page']}")
        print()
        print(p["text"][:1500])

        break

Preview of First Page with Text
Page Number : 4

ii
 
First Edition	
	
	
	
January 2026 Pausha 1947
PD 1500T SM
©	National Council of Educational  
Research and Training, 2026
` 145.00
ALL RIGHTS RESERVED
	 No part of this publication may be reproduced, stored in 
a retrieval system or transmitted, in any form or by any 
means, electronic, mechanical, photocopying, recording or 
otherwise without the prior permission of the publisher.
	 This book is sold subject to the condition that it shall not, 
by way of trade,  be lent,  re-sold, hired out or otherwise 
disposed of without the publisher’s consent, in any form of 
binding or cover other than that in which it is published.
	 The correct price of this publication is the price printed on 
this page, Any revised price indicated by a rubber stamp or 
by a sticker or by any other means is incorrect and should 
be unacceptable.
Publication Team
Head, Publication	
:	
M.V. Srinivasan 
Division
Chief Editor 	
:	
Bijnan Sutar
Chief Busines

In [35]:
import sys
import os

sys.path.append("../src")

In [14]:
from parser import PDFParser

In [36]:
PDF_PATH = "../data/pdf/Kaveri_English_Text_Book_Class_9.pdf"

parser = PDFParser(PDF_PATH)

print("Total Pages :", parser.get_total_pages())

Total Pages : 300


In [37]:
pages = parser.parse_pages()

print(len(pages))

300


In [38]:
import json
from pathlib import Path

output_dir = Path("../data")

pages_json = []

for p in pages:
    pages_json.append({
        "page": p["page"],
        "text": p["text"]
    })

with open(output_dir / "pages.json", "w", encoding="utf-8") as f:
    json.dump(
        pages_json,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", output_dir / "pages.json")

Saved: ../data/pages.json


In [39]:
import os

print(os.path.exists("../data/pages.json"))

True
